# 📚 02_数据融合与变形

> **核心能力**：多表连接 (JOIN)、长宽表互换 (pivot/melt)、笛卡尔积
>
> **开局心法**：先想清楚"我要按什么列拼表"和"是内连接还是左连接"，然后 `pd.merge` 一句搞定。长宽表切换别手写循环，用 `pivot_table` + `melt` 秒杀。

---

## 1️⃣ 数据拼接：让 A 表认识 B 表 (pd.merge 对应 SQL JOIN)

> **大白话口诀**：核心是找到 `on='牵线红娘的列名'`。`how=` 决定了如果这相亲配不成，是对不上的踢掉 (`inner`) 还是保留主表的人 (`left`)。

In [ ]:
import pandas as pd

# 1. 👈 [左连接 Left Join]：以主表(df_A)为主，从 df_B 把单价抄过来。（万能绝招）
# df_merged = pd.merge(df_A, df_B, on='硬件类型', how='left')

# 2. 🤝 [内连接 Inner Join]：只要双方都有的。比如找既在"暴利预警列表"里，又在我们数据库里的品牌。
# df_profitable = pd.merge(df_brands, df_thresholds, on='品牌', how='inner')

# 3. 🐍 [自连接 Self Join 的黑魔法]：同款电脑，用它高核显版的价格 减去 它的低核显版的价格。
#    通过 `suffixes` 自动帮你改名叫 '价格_高配', '价格_标配'
df_high = df[df['内存'] == 24]
df_low = df[df['内存'] == 16]
df_diff = pd.merge(df_high, df_low, on='机型', suffixes=('_高配', '_标配'))
# df_diff['厂家白赚你的钱'] = df_diff['价格_高配'] - df_diff['价格_标配']

### 📊 merge 的 4 种 JOIN 类型

| how 参数 | SQL 对标 | 含义 | 何时用 |
|---------|--------|------|--------|
| `'inner'` | INNER JOIN | 只要双方都有的 | 查"既在库存又在订单"的商品 |
| `'left'` | LEFT JOIN | 以左表为主，右表信息可有可无 | 给所有订单加上商品信息（万能） |
| `'right'` | RIGHT JOIN | 以右表为主（很少用） | 给所有商品加上最近订单 |
| `'outer'` | FULL OUTER JOIN | 两表的所有行都保留 | 找"只在A有"或"只在B有"的数据 |

---

### 🚨 坑点：merge 行数爆炸

#### 坑点 1：merge 后行数突然变多了（笛卡尔积炸弹）

```python
# 【场景】df_A 有 100 行，df_B 有 50 行
result = pd.merge(df_A, df_B, on='品牌')  # 期待 100 行，结果变成 5000 行！

# 【原因】
# 如果某个"品牌"在 A 里出现 10 次，在 B 里出现 5 次
# merge 时会两两配对，变成 10 × 5 = 50 行

# 【解决方案】
# 方案1：先去重。比如，B 表中每个品牌只保留一条
df_B_clean = df_B.drop_duplicates(subset=['品牌'], keep='first')
result = pd.merge(df_A, df_B_clean, on='品牌')

# 方案2：检查 merge 前两表中"品牌"列的重复情况
print(df_A['品牌'].value_counts())
print(df_B['品牌'].value_counts())
```

---

#### 坑点 2：merge 后列名冲突（都叫"价格"）

```python
# ❌ A 表和 B 表都有"价格"列，merge 时出错或产生奇怪的列名
result = pd.merge(df_A, df_B, on='商品ID')
# 结果：变成"价格_x"和"价格_y"（很丑陋）

# ✅ 用 suffixes 参数重命名
result = pd.merge(df_A, df_B, on='商品ID', suffixes=('_A表', '_B表'))
# 结果："价格_A表"和"价格_B表"（清晰明了）
```

---
## 2️⃣ 行列转换：长宽表互换 (pivot_table & melt)

> **大白话口诀**：
> *   `pivot_table` (长变宽)：把**某列里的值**变成**表头**（做做报表/给人看专用）。
> *   `melt` (宽变长)：把**一排表头**融化成**一列里的值**（存进数据库/画大屏图表专用）。
>
> ⚠️ **避坑指南**：如果你自己用代码 `pivot` 出了宽表，之后又要去 `melt`，中间务必加一句 `.rename_axis(None, axis=1)` 抹掉隐形的列名标签！

In [ ]:
# 1. 📈 [长表变宽表 (做报表)]：透视表 `pivot_table`
# 把 '季度' 里的名字 (Q1、Q2) 拆出来挂到表头，当新列名。
# 【SQL 视角】类似于复杂的 CASE WHEN sum。

df_wide = df.pivot_table(
    index='品牌',       # 横向的行标 (相当于 GROUP BY 品牌)
    columns='季度',     # 纵向的列标 (谁的内容挂在表头上)
    values='销售额',    # 中间的格子填什么数字
    aggfunc='sum',      # 重复的怎么算 (求和、平均)
    margins=True,       # 自动生成 "All" (总计) 这一行/列
    fill_value=0        # 如果有空格子，填上 0
)

# 🎉 打扫战场环节 (如果是给下面 melt 铺垫的话)：
df_wide = df_wide.reset_index()               # 把 '品牌' 从黑粗体索引变回普通列
df_wide = df_wide.rename_axis(None, axis=1)   # 把悬在表头上的 '季度' 标签擦掉！

In [ ]:
# 2. 📉 [宽表变长表 (存库)]：反透视融化 `melt`
# 把所有的 '季度' 比如 Q1, Q2 (除了'品牌'之外的所有列) 融化成两列。

df_long = df_wide.melt(
    id_vars=['品牌'],           # 固定不动的列 (保留当参照物的)
    var_name='季度_名称',       # 融化下来的那些表头 (Q1,Q2)，新列名叫什么？
    value_name='当季销售额'     # 融化下来的那些格子里的数字，新列名叫什么？
)

### 💡 从纯 SQL 角度理解 Pandas 透视表的"降维打击"

如果不用 `pivot_table`，要实现"行转列"（按地区或季度），纯 SQL 只能写无数个 `CASE WHEN` 交叉表，维护堪称地狱模式。

更可怕的是**求平均的致命陷阱 (AVG Bug)**：
* **求 SUM 时：** `SUM(CASE WHEN 季度='Q1' THEN 销售额 ELSE 0 END)` 没问题。
* **求 AVG 时：如果照抄加了 `ELSE 0`** ⬇️
  ```sql
  AVG(CASE WHEN 季度='Q1' THEN 销售额 ELSE 0 END) 
  ```
  👉 **大错特错！** 这会把非 Q1 的月份当做 `0` 并入计算，导致你的分母变大，平均值严重变小！
  
  **必须写成**：
  ```sql
  AVG(CASE WHEN 季度='Q1' THEN 销售额 ELSE NULL END) 
  ```
  或者直接省掉 ELSE。

但在 Pandas 里，这一切不需要写 CASE WHEN 狂魔版，只要：
```python
aggfunc=['sum', 'mean']  # 搞定！
```

### 🚨 坑点：pivot_table 与 melt 的陷阱

#### 坑点 1：pivot 之后再 melt，多出一个奇怪的列

```python
# ❌ 以下代码的问题
df_wide = df.pivot_table(...)  # 产生了一个"隐形的列标签"
df_long = df_wide.melt(id_vars=['品牌'], ...)  # 融化后可能多出一列叫"季度"（列名标签）

# ✅ 正确做法：pivot 之后立即擦掉列标签
df_wide = df.pivot_table(...)
df_wide = df_wide.reset_index()                 # ← 把行标签变回列
df_wide = df_wide.rename_axis(None, axis=1)     # ← 把列标签擦掉
df_long = df_wide.melt(...)
```

---

#### 坑点 2：aggfunc='mean' 时的平均值陷阱

```python
# 【场景】同个品牌在 Q1 出现 3 次，你想求平均价格
result = df.pivot_table(
    index='品牌',
    columns='季度',
    values='价格',
    aggfunc='mean'
)
# 结果：自动把同品牌同季度的重复记录求平均
# （不像 SQL 的"ELSE 0 陷阱"，Pandas 的 aggfunc 很聪明）
```

---
## 3️⃣ 全量沙盒：笛卡尔积 (Cross Join)

> **大白话口诀**：
> *   **Cross Join** = 把 A 表的每一行和 B 表的每一行**两两配对**，一个都不落下！
> *   结果行数 = `A的行数 × B的行数`。小心爆炸！

### 何时用
- 造测试数据：3 个城市 × 4 个季度 = 12 种可能组合
- 造全量基础表，后续 left join 上其他数据

In [ ]:
# 准备两张小表
df_cities = pd.DataFrame({'城市': ['北京', '上海', '深圳']})
df_quarters = pd.DataFrame({'季度': ['Q1', 'Q2', 'Q3', 'Q4']})

# =====================================================
# 方法一：Pandas 专属写法 (推荐) — `how='cross'`
# =====================================================
df_cross = pd.merge(df_cities, df_quarters, how='cross')
# 结果：3×4=12 行

# =====================================================
# 方法二：万能土法 — 先造一个临时 key 列，按 left 连再删掉
# 适合老旧版本 Pandas (不支持 cross) 或者 SQL
# =====================================================
df_cities['key'] = 1
df_quarters['key'] = 1
df_cross_2 = pd.merge(df_cities, df_quarters, on='key', how='left').drop('key', axis=1)

---
## 【SQL 对照】merge 全家桶

```sql
-- 1. LEFT JOIN
SELECT a.*, b.单价
FROM orders a
LEFT JOIN products b ON a.品牌 = b.品牌;

-- 2. INNER JOIN
SELECT a.*, b.预警状态
FROM our_brands a
INNER JOIN risk_list b ON a.品牌 = b.品牌;

-- 3. SELF JOIN（带 suffixes）
SELECT a.*, b.价格 AS 价格_高配
FROM products a
LEFT JOIN products b ON a.机型 = b.机型 AND b.内存 = 24;

-- 4. CROSS JOIN（笛卡尔积）
SELECT * FROM cities CROSS JOIN quarters;
-- 或老语法
SELECT * FROM cities, quarters;
```